# 05 - Model comparison

Scores the **identical** test rows (same bookings, same label `cancel-by-arrival`, hazard at
d = min(lead, 14)).

ROC-AUC (ranking), PR-AUC (ranking at low prevalence - the honest one here), Brier + reliability (calibration, because the
probabilities feed the cost decision), and confusion matrices (raw + normalised) at each model's cost-optimal threshold.

## 0 - Setup

In [1]:
from __future__ import annotations
import sys, time
from pathlib import Path
_t0=time.perf_counter()
def _step(m): print(f"  [{time.perf_counter()-_t0:5.2f}s] {m}", flush=True)
_here=Path.cwd().resolve()
while not (_here/"pyproject.toml").exists():
    if _here==_here.parent: raise RuntimeError("project root not found")
    _here=_here.parent
if str(_here) not in sys.path: sys.path.insert(0, str(_here))
import numpy as np, pandas as pd
import plotly.graph_objects as go, plotly.io as pio
from plotly.subplots import make_subplots
from sklearn.metrics import (roc_auc_score, average_precision_score, brier_score_loss,
                             roc_curve, precision_recall_curve, confusion_matrix)
from sklearn.calibration import calibration_curve
from src import color, figures_dir, tables_dir
import src.training as T, src.scoring as sc
pio.templates.default="plotly_white"
BRAND={n: color(n) for n in ["yellow","blue","green","orange","pink","purple","red"]}
MCOL={"logreg":BRAND["blue"],"xgboost":BRAND["orange"],"histgb":BRAND["green"],
      "hazard":BRAND["purple"],"baseline":"grey"}
FIG_DIR=figures_dir()/"05_comparison"; FIG_DIR.mkdir(parents=True, exist_ok=True)
TBL_DIR=tables_dir()/"05_comparison"; TBL_DIR.mkdir(parents=True, exist_ok=True)
RANDOM_STATE=42
_step("setup done (plotly).")

  [ 1.54s] setup done (plotly).


## 1 - Matched predictions + random baseline

`bakeoff_walk_forward` returns one row per test booking with every model's probability
on the same rows. The **random baseline** draws a hard cancel/not label ~ Bernoulli(p0)
at the test prevalence p0.

In [2]:
status = T.bakeoff_cache_status()   # 'fresh' | 'stale' | 'missing' (stale => a retrain / data refresh happened)
try:
    bake = T.load_bakeoff()          # fresh cache only; StaleArtifact / FileNotFound => recompute below
    _step(f"loaded cached bake-off ({len(bake):,} rows, cache '{status}'); delete Data/bakeoff_predictions.parquet to force recompute")
except (FileNotFoundError, T.StaleArtifact) as e:
    _step(f"computing matched bake-off (cache '{status}': {type(e).__name__}); fits 3 static + hazard per fold, heavy...")
    bake = T.bakeoff_walk_forward(n_folds=8, horizon_days=14, step_days=14, seed=RANDOM_STATE)
MODELS = ["logreg","xgboost","histgb","hazard"]
y = bake["y_true"].to_numpy()
P = {m: bake[f"p_{m}"].to_numpy() for m in MODELS}
p0 = float(y.mean())
rng = np.random.default_rng(RANDOM_STATE)
P["baseline"] = (rng.random(len(y)) < p0).astype(float)          # random Bernoulli(p0)
print(f"matched test rows: {len(y):,}  | test prevalence p0 = {p0:.3f} "
      f"(decision population; survivorship lowers it below the ~20% overall)")
print("  fairness check - all models scored on identical rows:", bake['y_true'].notna().all())

  [ 1.55s] fair matched bake-off (fits 3 static + hazard per fold; heavy)...


/Users/ruby.grambauer/Documents/DEV/OverbookingAnalyse/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/ruby.grambauer/Documents/DEV/OverbookingAnalyse/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/ruby.grambauer/Documents/DEV/OverbookingAnalyse/.venv/lib/python3.12/site-packages/sklearn/linear_model/_

matched test rows: 22,529  | test prevalence p0 = 0.125 (decision population; survivorship lowers it below the ~20% overall)
  fairness check - all models scored on identical rows: True


## 2 - Ranking: ROC-AUC and PR-AUC

ROC (rank quality overall) and Precision-Recall (rank quality where positives are
rare - the metric that actually matters at ~12% prevalence). No-skill references: the
ROC diagonal (AUC 0.5) and the PR line at p0.

In [3]:
fig = make_subplots(rows=1, cols=2, subplot_titles=("ROC curve", "Precision-Recall curve"))
for m in MODELS+["baseline"]:
    fpr,tpr,_ = roc_curve(y, P[m]); fig.add_scatter(x=fpr,y=tpr,mode="lines",name=m,legendgroup=m,
        line=dict(color=MCOL[m]), row=1,col=1)
    pr,rc,_ = precision_recall_curve(y, P[m]); fig.add_scatter(x=rc,y=pr,mode="lines",name=m,legendgroup=m,
        showlegend=False, line=dict(color=MCOL[m]), row=1,col=2)
fig.add_scatter(x=[0,1],y=[0,1],mode="lines",line=dict(color="black",dash="dot"),name="no-skill",row=1,col=1)
fig.add_scatter(x=[0,1],y=[p0,p0],mode="lines",line=dict(color="black",dash="dot"),showlegend=False,row=1,col=2)
fig.update_xaxes(title_text="FPR",row=1,col=1); fig.update_yaxes(title_text="TPR",row=1,col=1)
fig.update_xaxes(title_text="recall",row=1,col=2); fig.update_yaxes(title_text="precision",row=1,col=2)
fig.update_layout(title="Discrimination on the matched test set"); fig.show()

rank = pd.DataFrame({m: {"ROC_AUC": roc_auc_score(y,P[m]), "PR_AUC": average_precision_score(y,P[m]),
                        "Brier": brier_score_loss(y, np.clip(P[m],0,1))} for m in MODELS+["baseline"]}).T
display(rank.round(4))

,ROC_AUC,PR_AUC,Brier
logreg,0.7258,0.2336,0.1142
xgboost,0.7374,0.2631,0.1214
histgb,0.7467,0.2731,0.1037
hazard,0.7353,0.3000,0.0992
baseline,0.4983,0.1242,0.2180


## 3 - Calibration: reliability + Brier

The probabilities drive the overbooking math, so calibration is as important as
ranking. Reliability curves (predicted vs observed) + Brier (lower = better).
The random baseline's Brier equals the no-skill reference 2·p0·(1-p0).

In [4]:
fig = go.Figure()
for m in MODELS:
    fp,mp = calibration_curve(y, np.clip(P[m],0,1), n_bins=10, strategy="quantile")
    fig.add_scatter(x=mp,y=fp,mode="lines+markers",name=m,line=dict(color=MCOL[m]))
mx=max(np.clip(P[m],0,1).max() for m in MODELS)
fig.add_scatter(x=[0,mx],y=[0,mx],mode="lines",line=dict(color="grey",dash="dash"),name="perfect")
fig.update_layout(title="Reliability (pooled matched OOS)", xaxis_title="mean predicted", yaxis_title="observed freq")
fig.show()

fig = go.Figure(go.Bar(x=list(rank.index), y=rank["Brier"], marker_color=[MCOL[m] for m in rank.index]))
fig.add_hline(y=2*p0*(1-p0), line=dict(color="black",dash="dot"), annotation_text="no-skill 2p0(1-p0)")
fig.update_layout(title="Brier score (lower = better)", yaxis_title="Brier"); fig.show()

## 4 - Confusion matrices (raw + normalised) at cost-optimal thresholds

Each model gets its own **cost-optimal** threshold (walk vs empty-room asymmetry);
the baseline uses its random labels. Raw counts in the table, row-normalised rates as
heatmaps - so a model that just flags everything (or nothing) is obvious.

In [5]:
rows=[]; thr={}
for m in MODELS+["baseline"]:
    t = 0.5 if m=="baseline" else sc.cost_threshold_from_scores(y, P[m]); thr[m]=t
    pred = (P[m] >= t).astype(int)
    tn,fp,fn,tp = confusion_matrix(y, pred).ravel()
    prec = tp/(tp+fp) if tp+fp else 0.0; rec = tp/(tp+fn) if tp+fn else 0.0
    rows.append({"model":m,"threshold":round(t,3),"TP":tp,"FP":fp,"FN":fn,"TN":tn,
                 "precision":round(prec,3),"recall":round(rec,3),
                 "cost":sc.cost_at_threshold(y,P[m],t)["total_cost"]})
cm_tbl=pd.DataFrame(rows).set_index("model"); display(cm_tbl)

fig=make_subplots(rows=1, cols=len(MODELS)+1, subplot_titles=MODELS+["baseline"],
                  horizontal_spacing=0.04)
for i,m in enumerate(MODELS+["baseline"], start=1):
    pred=(P[m]>=thr[m]).astype(int); cm=confusion_matrix(y,pred)
    cmn=cm/cm.sum(axis=1, keepdims=True)
    txt=[[f"{cm[r,c]:,}<br>{cmn[r,c]:.0%}" for c in range(2)] for r in range(2)]
    fig.add_trace(go.Heatmap(z=cmn, x=["pred 0","pred 1"], y=["true 0","true 1"],
                  text=txt, texttemplate="%{text}", zmin=0, zmax=1, coloraxis="coloraxis"), row=1, col=i)
fig.update_layout(title="Confusion (colour = row-normalised rate; label = count + %)",
                  coloraxis=dict(colorscale="Blues")); fig.update_yaxes(autorange="reversed")
fig.show()

,threshold,TP,FP,FN,TN,precision,recall,cost
model,,,,,,,,
logreg,0.743,0,0,2807,19722,0.000,0.000,224560.0
xgboost,0.550,14,0,2793,19722,1.000,0.005,223440.0
histgb,0.671,12,3,2795,19719,0.800,0.004,224500.0
hazard,0.754,30,5,2777,19717,0.857,0.011,223660.0
baseline,0.500,338,2442,2469,17280,0.122,0.120,930120.0


## 5 - Selection

In [6]:
BRIER_TOL=0.005
cand={m:rank.loc[m] for m in MODELS}
best_brier=min(v["Brier"] for v in cand.values())
eligible=[m for m,v in cand.items() if v["Brier"]<=best_brier+BRIER_TOL]
winner=max(eligible, key=lambda m: cand[m]["PR_AUC"])
print("Brier-eligible:", eligible)
print(f"WINNER by PR-AUC among calibrated models: {winner}  "
      f"(PR-AUC={cand[winner]['PR_AUC']:.4f}, Brier={cand[winner]['Brier']:.4f})")
rank.to_csv(TBL_DIR/"model_ranking.csv"); cm_tbl.to_csv(TBL_DIR/"confusion.csv")

Brier-eligible: ['histgb', 'hazard']
WINNER by PR-AUC among calibrated models: hazard  (PR-AUC=0.3000, Brier=0.0992)


## 6 - What does the hazard buy us?

The static models give ONE score per booking; the hazard conditions on how far arrival is.
We test promotion the honest way: a **paired bootstrap CI** on the SAME rows for **PR-AUC**
(the metric that matters at ~12% prevalence) and **expected cost** - not a "mean(dAUC) > its
own std" heuristic. We also report **within-horizon AUC** (discrimination at a FIXED
days-until-arrival), because a pooled AUC is partly mechanical: it rewards the model for
knowing how close arrival is.

**Caveat on "near arrival":** here `d = min(lead, 14)` is the booking's LEAD TIME, so `d<=3`
is the **last-minute subpopulation** (booked <=3 days out - a distinct, low-cancellation
group), NOT a fixed booking re-scored 3 days before arrival. At `d=1` the single-window
survival product collapses to ~0, so its AUC is ~0.5 by construction. The time-resolved
"does risk rise toward arrival" proof lives in notebook 08 section 7.

In [ ]:
import src.model_eval as ME
best_static = max(MODELS[:-1], key=lambda m: rank.loc[m,"PR_AUC"])

# Promotion test: paired BOOTSTRAP CIs on the SAME rows (PR-AUC = the low-prevalence
# metric that matters here) + delta expected-cost - not the old mean(dAUC)>std heuristic.
rep = ME.promotion_report(bake, challenger="hazard", baseline=best_static)
print(f"hazard vs best static ({best_static})  [paired bootstrap, n={rep['n']:,}]")
print(f"  dPR-AUC  = {rep['delta_pr_auc']['delta']:+.4f}  CI {rep['delta_pr_auc']['ci']}  "
      f"(P better {rep['delta_pr_auc']['p_challenger_better']:.2f})")
print(f"  dROC-AUC = {rep['delta_roc_auc']['delta']:+.4f}  CI {rep['delta_roc_auc']['ci']}")
print(f"  cost: hazard {rep['cost_challenger']:,.0f} vs {best_static} {rep['cost_baseline']:,.0f} "
      f"(delta {rep['delta_cost']:+,.0f})")
print(f"  VERDICT: {'PROMOTE hazard' if rep['promote'] else 'HOLD'} - {rep['reason']}")

# HONEST per-booking skill: AUC at a FIXED horizon (strips the mechanical between-horizon
# separation that inflates the pooled AUC).
wh, wmean = ME.within_horizon_auc(bake, prob_col="p_hazard", day_col="d")
print(f"\nwithin-horizon (fixed-day) hazard AUC: n-weighted mean {wmean:.3f} vs pooled "
      f"{roc_auc_score(y, P['hazard']):.3f}  (pooled inflated by between-horizon separation)")
display(wh.round(3))

# 'Near arrival' CAVEAT: d = min(lead, 14) is the booking's LEAD TIME, so d<=3 is the
# LAST-MINUTE subpopulation (distinct, low-cancellation), NOT a fixed booking seen 3 days
# out. At d=1 the single-window survival product collapses to ~0 (AUC ~0.5 by construction).
d = bake["d"].to_numpy()
for label, mask in [("short-lead d<=3", d<=3), ("rest d>3", d>3)]:
    yy=y[mask]
    if len(np.unique(yy))<2: continue
    ah=roc_auc_score(yy,P["hazard"][mask]); a_s=roc_auc_score(yy,P[best_static][mask])
    print(f"  {label:16s} n={mask.sum():>6,}: hazard AUC {ah:.3f} vs {best_static} {a_s:.3f}  (delta {ah-a_s:+.3f})")

fig=go.Figure(go.Bar(x=wh['day'], y=wh['auc'], marker_color=BRAND['purple']))
fig.update_layout(title='Honest per-booking skill: hazard AUC at a fixed horizon',
                  xaxis_title='days until arrival (d)', yaxis_title='within-horizon AUC')
fig.update_xaxes(autorange='reversed'); fig.show()
print('Time-resolved per-horizon calibration lives in 08 section 7.')